In [3]:
# import os
# import re
# import glob
# import numpy as np
# import pandas as pd

# from collections import defaultdict
# from statsmodels.stats.multitest import multipletests

# # ============================================================
# # CONFIG
# # ============================================================

# INPUT_DIR = "/n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results"

# OUTPUT_DIR = "/n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed"
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# # Cell types to combine now
# CELLTYPES = ["Ast", "In", "Oli", "Opc"]

# # Expected number of chunks per cell type
# EXPECTED_CHUNKS = {
#     "Ast": 5,
#     "In": 30,
#     "Oli": 40,
#     "Opc": 5,
# }

# LOG2FC_THRESH = 0.25
# Q_THRESH = 0.05

# # ============================================================
# # FILENAME REGEX (must match your pipeline exactly)
# # ============================================================

# pattern = re.compile(
#     r"poisson_DE_results_PFC_(?P<celltype>\w+)_set(?P<set>\d+)_chunk(?P<chunk>\d+)_of_(?P<nchunks>\d+)\.csv"
# )

# # ============================================================
# # DISCOVER FILES + VERIFY COMPLETENESS
# # ============================================================

# files_by_cell_set = defaultdict(lambda: defaultdict(dict))

# for fname in os.listdir(INPUT_DIR):
#     m = pattern.match(fname)
#     if not m:
#         continue

#     cell = m.group("celltype")
#     if cell not in CELLTYPES:
#         continue

#     set_id = int(m.group("set"))
#     chunk  = int(m.group("chunk"))
#     nchunks = int(m.group("nchunks"))

#     files_by_cell_set[cell][set_id][chunk] = os.path.join(INPUT_DIR, fname)

# # Verify completeness
# for cell in CELLTYPES:
#     expected = EXPECTED_CHUNKS[cell]
#     if cell not in files_by_cell_set:
#         raise RuntimeError(f"{cell}: no files found at all")

#     for set_id, chunks in files_by_cell_set[cell].items():
#         missing = sorted(set(range(1, expected + 1)) - set(chunks.keys()))
#         if missing:
#             raise RuntimeError(
#                 f"{cell} set {set_id}: missing chunks {missing} "
#                 f"(expected {expected})"
#             )

# print("✅ All required chunks present for Ast / In / Oli / Opc")

# # ============================================================
# # LOAD + CONCATENATE CHUNKS
# # ============================================================

# dfs = []

# for cell in CELLTYPES:
#     for set_id, chunk_map in files_by_cell_set[cell].items():
#         for chunk_id in sorted(chunk_map):
#             df = pd.read_csv(chunk_map[chunk_id])

#             df = df[[
#                 "gene",
#                 "estimate_AD",
#                 "se_AD",
#                 "pval_AD",
#                 "n_cells"
#             ]].copy()

#             df["celltype"] = cell
#             df["set_id"] = set_id
#             df["chunk_id"] = chunk_id

#             dfs.append(df)

# all_df = pd.concat(dfs, ignore_index=True)

# # ============================================================
# # RECOMPUTE GLOBAL FDR PER CELL TYPE
# # ============================================================

# def recompute_fdr(df):
#     df = df.copy()

#     mask = df["pval_AD"].notna()
#     qvals = np.full(len(df), np.nan)

#     qvals[mask] = multipletests(
#         df.loc[mask, "pval_AD"].values,
#         method="fdr_bh"
#     )[1]

#     df["p_adj"] = qvals
#     df["log2FC"] = df["estimate_AD"] / np.log(2)
#     df["DEG"] = (
#         (df["p_adj"] < Q_THRESH) &
#         (df["log2FC"].abs() > LOG2FC_THRESH)
#     )
#     return df

# # ============================================================
# # WRITE COMBINED OUTPUTS
# # ============================================================

# for cell in CELLTYPES:
#     sub = all_df[all_df["celltype"] == cell].copy()

#     # Safety: chunks are disjoint, but guard anyway
#     sub = (
#         sub.sort_values("pval_AD")
#            .drop_duplicates(["set_id", "gene"], keep="first")
#     )

#     combined = recompute_fdr(sub)

#     out_path = os.path.join(
#         OUTPUT_DIR,
#         f"poisson_DE_results_PFC_{cell}_COMBINED.csv"
#     )

#     combined.to_csv(out_path, index=False)

#     print(
#         f"{cell}: genes={combined.shape[0]} | "
#         f"DEGs={(combined['DEG']).sum()} | "
#         f"written -> {out_path}"
#     )

# print("\n✅ DONE — chunk-aware DEG combination complete.")

✅ All required chunks present for Ast / In / Oli / Opc
Ast: genes=3894 | DEGs=83 | written -> /n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed/poisson_DE_results_PFC_Ast_COMBINED.csv
In: genes=6145 | DEGs=2 | written -> /n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed/poisson_DE_results_PFC_In_COMBINED.csv
Oli: genes=2862 | DEGs=29 | written -> /n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed/poisson_DE_results_PFC_Oli_COMBINED.csv
Opc: genes=4697 | DEGs=8 | written -> /n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed/poisson_DE_results_PFC_Opc_COMBINED.csv

✅ DONE — chunk-aware DEG combination complete.


In [8]:
# import os
# import re
# import shutil
# import numpy as np
# import pandas as pd

# from collections import defaultdict
# from statsmodels.stats.multitest import multipletests

# # ============================================================
# # CONFIG
# # ============================================================

# INPUT_DIR = "/n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results"

# OUTPUT_DIR = os.path.join(INPUT_DIR, "Processed")
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# # Cell types that require chunk-combination
# CHUNKED_CELLTYPES = ["Ast", "In", "Oli", "Opc", "Ex"]

# # Expected number of chunks per set
# EXPECTED_CHUNKS = {
#     "Ast": 5,
#     "In": 30,
#     "Oli": 40,
#     "Opc": 5,
#     "Ex": 30,
# }

# # Microglia is a single file
# MIC_FILE = "poisson_DE_results_PFC_Mic.csv"

# LOG2FC_THRESH = 0.25
# Q_THRESH = 0.05

# # ============================================================
# # FILENAME REGEX (must match pipeline exactly)
# # ============================================================

# pattern = re.compile(
#     r"poisson_DE_results_PFC_(?P<celltype>\w+)_set(?P<set>\d+)_chunk(?P<chunk>\d+)_of_(?P<nchunks>\d+)\.csv"
# )

# # ============================================================
# # STEP 1 — COPY + RENAME MICROGLIA (NO PROCESSING)
# # ============================================================

# mic_src = os.path.join(INPUT_DIR, MIC_FILE)

# mic_dst = os.path.join(
#     OUTPUT_DIR,
#     "poisson_DE_results_PFC_Mic_COMBINED.csv"
# )

# if not os.path.exists(mic_src):
#     raise FileNotFoundError(f"Missing Microglia file: {mic_src}")

# shutil.copyfile(mic_src, mic_dst)

# print(f"✅ Microglia copied unchanged → {mic_dst}")

# # ============================================================
# # STEP 2 — DISCOVER + VERIFY CHUNKED FILES
# # ============================================================

# files_by_cell_set = defaultdict(lambda: defaultdict(dict))

# for fname in os.listdir(INPUT_DIR):
#     m = pattern.match(fname)
#     if not m:
#         continue

#     cell = m.group("celltype")
#     if cell not in CHUNKED_CELLTYPES:
#         continue

#     set_id  = int(m.group("set"))
#     chunk   = int(m.group("chunk"))
#     nchunks = int(m.group("nchunks"))

#     files_by_cell_set[cell][set_id][chunk] = os.path.join(INPUT_DIR, fname)

# # Verify completeness explicitly
# for cell in CHUNKED_CELLTYPES:
#     expected = EXPECTED_CHUNKS[cell]

#     if cell not in files_by_cell_set:
#         raise RuntimeError(f"{cell}: no files found at all")

#     for set_id, chunks in files_by_cell_set[cell].items():
#         missing = sorted(set(range(1, expected + 1)) - set(chunks.keys()))
#         if missing:
#             raise RuntimeError(
#                 f"{cell} set {set_id}: missing chunks {missing} "
#                 f"(expected {expected})"
#             )

# print("✅ All required chunks present for Ast / In / Oli / Opc / Ex")

# # ============================================================
# # STEP 3 — LOAD + CONCATENATE ALL CHUNKS
# # ============================================================

# dfs = []

# for cell in CHUNKED_CELLTYPES:
#     for set_id, chunk_map in files_by_cell_set[cell].items():
#         for chunk_id in sorted(chunk_map):
#             df = pd.read_csv(chunk_map[chunk_id])

#             df = df[[
#                 "gene",
#                 "estimate_AD",
#                 "se_AD",
#                 "pval_AD",
#                 "n_cells"
#             ]].copy()

#             df["celltype"] = cell
#             df["set_id"] = set_id
#             df["chunk_id"] = chunk_id

#             dfs.append(df)

# all_df = pd.concat(dfs, ignore_index=True)

# # ============================================================
# # STEP 4 — RECOMPUTE GLOBAL FDR PER CELL TYPE
# # ============================================================

# def recompute_fdr(df):
#     df = df.copy()

#     mask = df["pval_AD"].notna()
#     qvals = np.full(len(df), np.nan)

#     qvals[mask] = multipletests(
#         df.loc[mask, "pval_AD"].values,
#         method="fdr_bh"
#     )[1]

#     df["p_adj"] = qvals
#     df["log2FC"] = df["estimate_AD"] / np.log(2)
#     df["DEG"] = (
#         (df["p_adj"] < Q_THRESH) &
#         (df["log2FC"].abs() > LOG2FC_THRESH)
#     )
#     return df

# # ============================================================
# # STEP 5 — WRITE COMBINED OUTPUTS
# # ============================================================

# for cell in CHUNKED_CELLTYPES:
#     sub = all_df[all_df["celltype"] == cell].copy()

#     # Safety: chunks should be disjoint, but guard anyway
#     sub = (
#         sub.sort_values("pval_AD")
#            .drop_duplicates(["set_id", "gene"], keep="first")
#     )

#     combined = recompute_fdr(sub)

#     out_path = os.path.join(
#         OUTPUT_DIR,
#         f"poisson_DE_results_PFC_{cell}_COMBINED.csv"
#     )

#     combined.to_csv(out_path, index=False)

#     print(
#         f"{cell}: genes={combined.shape[0]} | "
#         f"DEGs={combined['DEG'].sum()} | "
#         f"written → {out_path}"
#     )

# print("\n✅ DONE — Ex + Mic + all other cell types processed safely.")

✅ Microglia copied unchanged → /n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed/poisson_DE_results_PFC_Mic_COMBINED.csv
✅ All required chunks present for Ast / In / Oli / Opc / Ex
Ast: genes=3894 | DEGs=83 | written → /n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed/poisson_DE_results_PFC_Ast_COMBINED.csv
In: genes=6145 | DEGs=2 | written → /n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed/poisson_DE_results_PFC_In_COMBINED.csv
Oli: genes=2862 | DEGs=29 | written → /n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed/poisson_DE_results_PFC_Oli_COMBINED.csv
Opc: genes=4697 | DEGs=8 | written → /n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed/poisson_DE_results_PFC_Opc_COMBINED.csv
Ex: genes=23906 | DEGs=113 | written → /n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed/poisson_DE_results_PFC_Ex_COMBINED.csv

✅ 

In [2]:
import os
import re
import shutil
import numpy as np
import pandas as pd
import math  # <-- FIX: use Python math for erf

from collections import defaultdict
from statsmodels.stats.multitest import multipletests

# ============================================================
# CONFIG
# ============================================================

INPUT_DIR = "/n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results"

OUTPUT_DIR = os.path.join(INPUT_DIR, "Processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Cell types that require chunk-combination
CHUNKED_CELLTYPES = ["Ast", "In", "Oli", "Opc", "Ex"]

# Expected number of chunks per set
EXPECTED_CHUNKS = {
    "Ast": 5,
    "In": 30,
    "Oli": 40,
    "Opc": 5,
    "Ex": 30,
}

# Microglia is a single file
MIC_FILE = "poisson_DE_results_PFC_Mic.csv"

LOG2FC_THRESH = 0.25
Q_THRESH = 0.05

# ============================================================
# FILENAME REGEX (must match pipeline exactly)
# ============================================================

pattern = re.compile(
    r"poisson_DE_results_PFC_(?P<celltype>\w+)_set(?P<set>\d+)_chunk(?P<chunk>\d+)_of_(?P<nchunks>\d+)\.csv"
)

# ============================================================
# STEP 1 — COPY + RENAME MICROGLIA (NO PROCESSING)
# ============================================================

mic_src = os.path.join(INPUT_DIR, MIC_FILE)

mic_dst = os.path.join(
    OUTPUT_DIR,
    "poisson_DE_results_PFC_Mic_COMBINED.csv"
)

if not os.path.exists(mic_src):
    raise FileNotFoundError(f"Missing Microglia file: {mic_src}")

shutil.copyfile(mic_src, mic_dst)

print(f"✅ Microglia copied unchanged → {mic_dst}")

# ============================================================
# STEP 2 — DISCOVER + VERIFY CHUNKED FILES
# ============================================================

files_by_cell_set = defaultdict(lambda: defaultdict(dict))

for fname in os.listdir(INPUT_DIR):
    m = pattern.match(fname)
    if not m:
        continue

    cell = m.group("celltype")
    if cell not in CHUNKED_CELLTYPES:
        continue

    set_id  = int(m.group("set"))
    chunk   = int(m.group("chunk"))
    nchunks = int(m.group("nchunks"))

    files_by_cell_set[cell][set_id][chunk] = os.path.join(INPUT_DIR, fname)

# Verify completeness explicitly
for cell in CHUNKED_CELLTYPES:
    expected = EXPECTED_CHUNKS[cell]

    if cell not in files_by_cell_set:
        raise RuntimeError(f"{cell}: no files found at all")

    for set_id, chunks in files_by_cell_set[cell].items():
        missing = sorted(set(range(1, expected + 1)) - set(chunks.keys()))
        if missing:
            raise RuntimeError(
                f"{cell} set {set_id}: missing chunks {missing} "
                f"(expected {expected})"
            )

print("✅ All required chunks present for Ast / In / Oli / Opc / Ex")

# ============================================================
# STEP 3 — LOAD + CONCATENATE ALL CHUNKS
# ============================================================

dfs = []

for cell in CHUNKED_CELLTYPES:
    for set_id, chunk_map in files_by_cell_set[cell].items():
        for chunk_id in sorted(chunk_map):
            df = pd.read_csv(chunk_map[chunk_id])

            df = df[[
                "gene",
                "estimate_AD",
                "se_AD",
                "pval_AD",
                "n_cells"
            ]].copy()

            df["celltype"] = cell
            df["set_id"] = set_id
            df["chunk_id"] = chunk_id

            dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)

# ============================================================
# STEP 4 — RECOMPUTE GLOBAL FDR PER CELL TYPE
# ============================================================

def recompute_fdr(df):
    df = df.copy()

    mask = df["pval_AD"].notna()
    qvals = np.full(len(df), np.nan)

    qvals[mask] = multipletests(
        df.loc[mask, "pval_AD"].values,
        method="fdr_bh"
    )[1]

    df["p_adj"] = qvals
    df["log2FC"] = df["estimate_AD"] / np.log(2)
    df["DEG"] = (
        (df["p_adj"] < Q_THRESH) &
        (df["log2FC"].abs() > LOG2FC_THRESH)
    )
    return df

# ============================================================
# STEP 5 — WRITE COMBINED OUTPUTS
# ============================================================

for cell in CHUNKED_CELLTYPES:
    sub = all_df[all_df["celltype"] == cell].copy()

    # Safety: chunks should be disjoint, but guard anyway
    sub = (
        sub.sort_values("pval_AD")
           .drop_duplicates(["set_id", "gene"], keep="first")
    )

    if cell == "Ex":
        # ------------------------------------------------------------
        # FIX: meta-analysis across Ex sets (disjoint donors)
        # One row per gene in final output
        # ------------------------------------------------------------
        sub = sub.dropna(subset=["gene", "estimate_AD", "se_AD"])
        sub = sub[sub["se_AD"] > 0].copy()

        # Fixed-effect inverse-variance meta-analysis per gene
        def _meta_one(g):
            beta = g["estimate_AD"].to_numpy(dtype=float)
            se = g["se_AD"].to_numpy(dtype=float)
            w = 1.0 / (se ** 2)

            beta_hat = (w * beta).sum() / w.sum()
            se_hat = 1.0 / np.sqrt(w.sum())
            z = beta_hat / se_hat

            # two-sided p-value from standard normal using erf
            # Phi(z) = 0.5 * (1 + erf(z / sqrt(2)))
            p = 2.0 * (1.0 - (0.5 * (1.0 + math.erf(abs(z) / math.sqrt(2.0)))))

            return pd.Series({
                "estimate_AD": beta_hat,
                "se_AD": se_hat,
                "pval_AD": p,
                "n_cells": g["n_cells"].sum(),
                "n_sets_used": g["set_id"].nunique()
            })

        combined = (
            sub.groupby("gene", as_index=False)
               .apply(lambda g: _meta_one(g))
               .reset_index()
        )
        if "index" in combined.columns:
            combined = combined.drop(columns=["index"])

        combined["celltype"] = "Ex"
        combined["set_id"] = "META"
        combined["chunk_id"] = "META"

        combined = recompute_fdr(combined)

    else:
        combined = recompute_fdr(sub)

    out_path = os.path.join(
        OUTPUT_DIR,
        f"poisson_DE_results_PFC_{cell}_COMBINED.csv"
    )

    combined.to_csv(out_path, index=False)

    print(
        f"{cell}: genes={combined.shape[0]} | "
        f"DEGs={combined['DEG'].sum()} | "
        f"written → {out_path}"
    )

print("\n✅ DONE — Ex + Mic + all other cell types processed safely.")

✅ Microglia copied unchanged → /n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed/poisson_DE_results_PFC_Mic_COMBINED.csv
✅ All required chunks present for Ast / In / Oli / Opc / Ex
Ast: genes=3894 | DEGs=83 | written → /n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed/poisson_DE_results_PFC_Ast_COMBINED.csv
In: genes=6145 | DEGs=2 | written → /n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed/poisson_DE_results_PFC_In_COMBINED.csv
Oli: genes=2862 | DEGs=29 | written → /n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed/poisson_DE_results_PFC_Oli_COMBINED.csv
Opc: genes=4697 | DEGs=8 | written → /n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed/poisson_DE_results_PFC_Opc_COMBINED.csv
Ex: genes=9012 | DEGs=40 | written → /n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed/poisson_DE_results_PFC_Ex_COMBINED.csv

✅ DO

/tmp/ipykernel_3368857/2918821144.py:196: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sub.groupby("gene", as_index=False)


In [3]:
# Predictor vs Random Effect DEG for 427 Mathys

import os
import pandas as pd
import numpy as np
import joblib
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# ============================
# Paths
# ============================
deg_dir = "/n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed"
model_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"

out_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/DEG_Predictor_Correlations_427"
os.makedirs(out_dir, exist_ok=True)

cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

results = []

# ============================
# Loop per cell type
# ============================
for ct in cell_types:
    print(f"\n=== {ct} ===")

    deg_path = os.path.join(deg_dir, f"poisson_DE_results_PFC_{ct}_COMBINED.csv")
    deg = pd.read_csv(deg_path)

    deg["gene"] = deg["gene"].str.strip()
    deg = deg.sort_values("p_adj", ascending=True)
    deg["deg_rank"] = np.arange(1, len(deg) + 1)

    all_split_data = {}
    all_features = set()

    for split in range(1, 6):
        model_path = os.path.join(model_dir, ct, f"split_{split}", "maximal_classifier.joblib")
        model = joblib.load(model_path)

        feats = model.feature_names_in_
        imps = model.feature_importances_

        all_features.update(feats)
        all_split_data[split] = dict(zip(feats, imps))

    rows = []
    for gene in sorted(all_features):
        norm_imps = []

        for split in range(1, 6):
            imp = all_split_data[split].get(gene, 0.0)
            split_imps = np.array(list(all_split_data[split].values()))
            denom = split_imps[split_imps > 0].sum()

            norm_imps.append(imp / denom if imp > 0 and denom > 0 else 0.0)

        rows.append({
            "gene": gene.strip(),
            "mean_importance": np.mean(norm_imps)
        })

    imp_df = pd.DataFrame(rows)
    imp_df["pred_rank"] = imp_df["mean_importance"].rank(ascending=False)

    merged = deg.merge(imp_df, on="gene", how="inner")

    rho, p = spearmanr(merged["deg_rank"], merged["pred_rank"])

    print(f"Spearman rho = {rho:.3f}, raw p = {p:.2e}")

    results.append({
        "cell_type": ct,
        "spearman_rho": rho,
        "p_value": p,
        "n_genes": merged.shape[0]
    })

    merged.to_csv(
        os.path.join(out_dir, f"{ct}_deg_predictor_rank_table.csv"),
        index=False
    )

# ============================
# Save summary + FDR correction
# ============================
summary = pd.DataFrame(results)

summary["p_value_fdr"] = multipletests(
    summary["p_value"], method="fdr_bh"
)[1]

summary.to_csv(
    os.path.join(out_dir, "DEG_predictor_spearman_summary.csv"),
    index=False
)

# ---------- PRINT FDR-ADJUSTED RESULTS ----------
print("\n=== FDR-adjusted Predictor–DEG correlations ===")
print(
    summary[["cell_type", "spearman_rho", "p_value", "p_value_fdr", "n_genes"]]
)

print("\nAll correlations complete.")


=== Ast ===
Spearman rho = 0.065, raw p = 5.42e-04

=== Mic ===
Spearman rho = 0.196, raw p = 2.90e-13

=== In ===
Spearman rho = 0.018, raw p = 1.89e-01

=== Oli ===
Spearman rho = 0.114, raw p = 3.01e-04

=== Opc ===
Spearman rho = 0.095, raw p = 1.62e-08

=== Ex ===
Spearman rho = 0.060, raw p = 3.54e-07

=== FDR-adjusted Predictor–DEG correlations ===
  cell_type  spearman_rho       p_value   p_value_fdr  n_genes
0       Ast      0.065306  5.420799e-04  6.504959e-04     2802
1       Mic      0.195683  2.898769e-13  1.739261e-12     1367
2        In      0.018361  1.887293e-01  1.887293e-01     5126
3       Oli      0.114061  3.012743e-04  4.519114e-04     1000
4       Opc      0.094540  1.620651e-08  4.861953e-08     3556
5        Ex      0.059944  3.544918e-07  7.089835e-07     7205

All correlations complete.


In [10]:
# # Predictor vs DEG (423) — overlap % + Fisher OR + FDR across cell types
# import os
# import numpy as np
# import pandas as pd
# import joblib
# from scipy.stats import fisher_exact
# from statsmodels.stats.multitest import multipletests

# # ============================
# # Paths
# # ============================
# deg423_dir  = "/n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed"
# model_dir   = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
# out_dir     = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/DEG423_Predictor_Overlap"
# os.makedirs(out_dir, exist_ok=True)

# cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

# # Predictor definition
# MIN_SPLITS = 2

# # DEG definition
# P_ADJ_CUTOFF = 0.05
# ABS_LOG2FC_CUTOFF = 0.25

# rows = []

# for ct in cell_types:
#     print(f"\n=== {ct} ===")

#     # ---------- Load 423 DEG ----------
#     deg_path = os.path.join(deg_dir, f"poisson_DE_results_PFC_{ct}_COMBINED.csv")
#     deg = pd.read_csv(deg_path)

#     deg["gene"] = deg["gene"].astype(str).str.strip()
#     deg["p_adj"] = pd.to_numeric(deg["p_adj"], errors="coerce")
#     deg["log2FC"] = pd.to_numeric(deg["log2FC"], errors="coerce")

#     # DEG-tested universe (genes present in DEG table with a defined log2FC + p_adj)
#     deg_tested = deg.dropna(subset=["p_adj", "log2FC"])
#     tested_genes = set(deg_tested["gene"])

#     # Significant DEGs
#     deg_sig = deg_tested[
#         (deg_tested["p_adj"] < P_ADJ_CUTOFF) &
#         (deg_tested["log2FC"].abs() > ABS_LOG2FC_CUTOFF)
#     ]
#     deg_sig_genes = set(deg_sig["gene"])

#     # ---------- Load predictors (stable predictors: nonzero in >=2 splits) ----------
#     gene_counts = {}
#     for split in range(1, 6):
#         model_path = os.path.join(model_dir, ct, f"split_{split}", "maximal_classifier.joblib")
#         model = joblib.load(model_path)

#         for g, imp in zip(model.feature_names_in_, model.feature_importances_):
#             if imp > 0:
#                 g = str(g).strip()
#                 gene_counts[g] = gene_counts.get(g, 0) + 1

#     predictors = {g for g, c in gene_counts.items() if c >= MIN_SPLITS}

#     # Restrict predictors to DEG-tested universe
#     predictors_tested = predictors & tested_genes

#     # Overlap
#     overlap = predictors_tested & deg_sig_genes

#     # Percent overlap / #predictors_tested
#     overlap_pct = (len(overlap) / len(predictors_tested)) if len(predictors_tested) > 0 else np.nan

#     # ---------- Fisher enrichment in DEG-tested universe ----------
#     # a = overlap
#     # b = predictors_tested not DEG
#     # c = DEG not predictors_tested
#     # d = neither
#     a = len(overlap)
#     b = len(predictors_tested) - a
#     c = len(deg_sig_genes - predictors_tested)
#     d = len(tested_genes) - (a + b + c)

#     # Defensive: if d < 0, something is inconsistent about universe definitions
#     if d < 0:
#         raise ValueError(f"Negative d for {ct}. Check universe definitions.")

#     OR, p = fisher_exact([[a, b], [c, d]], alternative="two-sided")

#     print(f"tested_genes={len(tested_genes)} | predictors_tested={len(predictors_tested)} | deg_sig={len(deg_sig_genes)}")
#     print(f"overlap={a} | overlap_pct={overlap_pct:.4f} | OR={OR:.3f} | raw p={p:.2e}")

#     # Save overlapping genes for this cell type
#     pd.DataFrame({"gene": sorted(overlap)}).to_csv(
#         os.path.join(out_dir, f"{ct}_predictor_DEG423_overlap_genes.csv"),
#         index=False
#     )

#     rows.append({
#         "cell_type": ct,
#         "n_tested_genes": len(tested_genes),
#         "n_predictors_total": len(predictors),
#         "n_predictors_tested": len(predictors_tested),
#         "n_deg_sig": len(deg_sig_genes),
#         "n_overlap": a,
#         "overlap_pct_of_predictors_tested": overlap_pct,
#         "odds_ratio": OR,
#         "p_value": p
#     })

# # ---------- Summary + FDR across cell types ----------
# summary = pd.DataFrame(rows)
# summary["p_value_fdr"] = multipletests(summary["p_value"], method="fdr_bh")[1]

# summary.to_csv(os.path.join(out_dir, "Predictor_vs_DEG423_overlap_fisher_summary.csv"), index=False)

# print("\n=== FDR-adjusted Predictor–DEG(423) overlap enrichment (Fisher) ===")
# print(summary[[
#     "cell_type",
#     "n_predictors_tested",
#     "n_deg_sig",
#     "n_overlap",
#     "overlap_pct_of_predictors_tested",
#     "odds_ratio",
#     "p_value",
#     "p_value_fdr"
# ]])


=== Ast ===
tested_genes=3894 | predictors_tested=158 | deg_sig=83
overlap=12 | overlap_pct=0.0759 | OR=4.243 | raw p=1.04e-04

=== Mic ===
tested_genes=2417 | predictors_tested=440 | deg_sig=15
overlap=9 | overlap_pct=0.0205 | OR=6.860 | raw p=3.60e-04

=== In ===
tested_genes=6145 | predictors_tested=102 | deg_sig=2
overlap=0 | overlap_pct=0.0000 | OR=0.000 | raw p=1.00e+00

=== Oli ===
tested_genes=2862 | predictors_tested=583 | deg_sig=29
overlap=12 | overlap_pct=0.0206 | OR=2.796 | raw p=9.08e-03

=== Opc ===
tested_genes=4697 | predictors_tested=788 | deg_sig=8
overlap=1 | overlap_pct=0.0013 | OR=0.708 | raw p=1.00e+00

=== Ex ===
tested_genes=9012 | predictors_tested=157 | deg_sig=82
overlap=10 | overlap_pct=0.0637 | OR=8.298 | raw p=1.41e-06

=== FDR-adjusted Predictor–DEG(423) overlap enrichment (Fisher) ===
  cell_type  n_predictors_tested  n_deg_sig  n_overlap  \
0       Ast                  158         83         12   
1       Mic                  440         15          9

In [4]:
# Predictor vs DEG (427) — overlap % + Fisher OR + FDR across cell types
import os
import numpy as np
import pandas as pd
import joblib
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

# ============================
# Paths
# ============================
deg423_dir  = "/n/scratch/users/a/adm808/Downloads/Revision/Maximal_Data_DEG_results/Processed"
model_dir   = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
out_dir     = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/DEG423_Predictor_Overlap"
os.makedirs(out_dir, exist_ok=True)

cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

# Predictor definition
MIN_SPLITS = 2

# DEG definition
P_ADJ_CUTOFF = 0.05
ABS_LOG2FC_CUTOFF = 0.25

rows = []

for ct in cell_types:
    print(f"\n=== {ct} ===")

    # ---------- Load 423 DEG ----------
    deg_path = os.path.join(deg_dir, f"poisson_DE_results_PFC_{ct}_COMBINED.csv")
    deg = pd.read_csv(deg_path)

    deg["gene"] = deg["gene"].astype(str).str.strip()
    deg["p_adj"] = pd.to_numeric(deg["p_adj"], errors="coerce")
    deg["log2FC"] = pd.to_numeric(deg["log2FC"], errors="coerce")

    # DEG-tested universe (genes present in DEG table with a defined log2FC + p_adj)
    deg_tested = deg.dropna(subset=["p_adj", "log2FC"])
    tested_genes = set(deg_tested["gene"])

    # Significant DEGs
    deg_sig = deg_tested[
        (deg_tested["p_adj"] < P_ADJ_CUTOFF) &
        (deg_tested["log2FC"].abs() > ABS_LOG2FC_CUTOFF)
    ]
    deg_sig_genes = set(deg_sig["gene"])

    # ---------- Load predictors (stable predictors: nonzero in >=2 splits) ----------
    gene_counts = {}
    for split in range(1, 6):
        model_path = os.path.join(model_dir, ct, f"split_{split}", "maximal_classifier.joblib")
        model = joblib.load(model_path)

        for g, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                g = str(g).strip()
                gene_counts[g] = gene_counts.get(g, 0) + 1

    predictors = {g for g, c in gene_counts.items() if c >= MIN_SPLITS}

    # Restrict predictors to DEG-tested universe
    predictors_tested = predictors & tested_genes

    # Overlap
    overlap = predictors_tested & deg_sig_genes

    # Percent overlap / #predictors_tested
    overlap_pct = (len(overlap) / len(predictors_tested)) if len(predictors_tested) > 0 else np.nan

    # ---------- Fisher enrichment in DEG-tested universe ----------
    # a = overlap
    # b = predictors_tested not DEG
    # c = DEG not predictors_tested
    # d = neither
    a = len(overlap)
    b = len(predictors_tested) - a
    c = len(deg_sig_genes - predictors_tested)
    d = len(tested_genes) - (a + b + c)

    # Defensive: if d < 0, something is inconsistent about universe definitions
    if d < 0:
        raise ValueError(f"Negative d for {ct}. Check universe definitions.")

    OR, p = fisher_exact([[a, b], [c, d]], alternative="two-sided")

    print(f"tested_genes={len(tested_genes)} | predictors_tested={len(predictors_tested)} | deg_sig={len(deg_sig_genes)}")
    print(f"overlap={a} | overlap_pct={overlap_pct:.4f} | OR={OR:.3f} | raw p={p:.2e}")

    # Save overlapping genes for this cell type
    pd.DataFrame({"gene": sorted(overlap)}).to_csv(
        os.path.join(out_dir, f"{ct}_predictor_DEG423_overlap_genes.csv"),
        index=False
    )

    rows.append({
        "cell_type": ct,
        "n_tested_genes": len(tested_genes),
        "n_predictors_total": len(predictors),
        "n_predictors_tested": len(predictors_tested),
        "n_deg_sig": len(deg_sig_genes),
        "n_overlap": a,
        "overlap_pct_of_predictors_tested": overlap_pct,
        "odds_ratio": OR,
        "p_value": p
    })

# ---------- Summary + FDR across cell types ----------
summary = pd.DataFrame(rows)
summary["p_value_fdr"] = multipletests(summary["p_value"], method="fdr_bh")[1]

summary.to_csv(os.path.join(out_dir, "Predictor_vs_DEG423_overlap_fisher_summary.csv"), index=False)

print("\n=== FDR-adjusted Predictor–DEG(423) overlap enrichment (Fisher) ===")
print(summary[[
    "cell_type",
    "n_predictors_tested",
    "n_deg_sig",
    "n_overlap",
    "overlap_pct_of_predictors_tested",
    "odds_ratio",
    "p_value",
    "p_value_fdr"
]])


=== Ast ===
tested_genes=3894 | predictors_tested=158 | deg_sig=83
overlap=12 | overlap_pct=0.0759 | OR=4.243 | raw p=1.04e-04

=== Mic ===
tested_genes=2417 | predictors_tested=440 | deg_sig=15
overlap=9 | overlap_pct=0.0205 | OR=6.860 | raw p=3.60e-04

=== In ===
tested_genes=6145 | predictors_tested=102 | deg_sig=2
overlap=0 | overlap_pct=0.0000 | OR=0.000 | raw p=1.00e+00

=== Oli ===
tested_genes=2862 | predictors_tested=583 | deg_sig=29
overlap=12 | overlap_pct=0.0206 | OR=2.796 | raw p=9.08e-03

=== Opc ===
tested_genes=4697 | predictors_tested=788 | deg_sig=8
overlap=1 | overlap_pct=0.0013 | OR=0.708 | raw p=1.00e+00

=== Ex ===
tested_genes=9012 | predictors_tested=157 | deg_sig=40
overlap=4 | overlap_pct=0.0255 | OR=6.405 | raw p=4.97e-03

=== FDR-adjusted Predictor–DEG(423) overlap enrichment (Fisher) ===
  cell_type  n_predictors_tested  n_deg_sig  n_overlap  \
0       Ast                  158         83         12   
1       Mic                  440         15          9 